### RUN IN TPU ~4:30:00 to process 1998-2026




In [ ]:
# CELL 0: Set Run Parameters
import os

os.environ["CACHE_LOOKBACK"] = "10"
os.environ["CACHE_START_DATE"] = "2026-01-01"

In [ ]:
# CELL 1
import time

start_time = time.time()

In [ ]:
# CELL 2: Setup
import sys, os
from pathlib import Path

RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    rl_root = Path(
        "/content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2"
    )
else:
    rl_root = Path("C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR_v2")

if str(rl_root) not in sys.path:
    sys.path.append(str(rl_root))
os.chdir(rl_root)

Mounted at /content/drive


In [ ]:
# CELL 3: Load Data & Initialize Cache
from core.settings import TradingConfig, CacheConfig
from core.paths import OUTPUT_DIR
from data_pipeline.loader import load_processed_data
from data_pipeline.utils import get_master_trading_calendar
from data_pipeline.screener import UniverseScreener
from data_pipeline.cache import CheckpointAlphaCache  # Extracted from original notebook

print("Loading preprocessed data...")
df_ohlcv, macro_df, features_df = load_processed_data()
config = TradingConfig()

# Reconstruct wide df_close dynamically and sync calendar
df_close = df_ohlcv["Adj Close"].unstack(level=0).sort_index()
trading_calendar = get_master_trading_calendar(df_ohlcv, config.calendar_ticker)

screener = UniverseScreener(
    df_close=df_close,
    features_df=features_df,
    macro_df=macro_df,
    trading_calendar=trading_calendar,
    config=config,
)

NOTEBOOKS_RLVR_ROOT: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2
PROJECT_ROOT: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks
GLOBAL_DATA_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/data
GLOBAL_PROCESSED_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/data/processed
LOCAL_DATA_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2/data
OUTPUT_DIR: /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2/output

Loading preprocessed data...


In [ ]:
# CELL 4: Build Cache
cache_file_path = OUTPUT_DIR / CacheConfig.get_filename()
cache = CheckpointAlphaCache(
    screener=screener, config=config, lookbacks=[CacheConfig.LOOKBACK]
)

cache.build(
    start_date=CacheConfig.START_DATE,
    end_date=None,
    checkpoint_path=cache_file_path,
    save_every_n_days=20,
)

[RESUME] Found existing cache file with 106 processed dates.
  Existing index range: 2026-01-02 to 2026-06-04
[INFO] Building AlphaCache with automatic resumption support:
  - Start Date:               2026-01-01
  - End Date Limit:           None (Processing all available dates)
  - Total target dates:       118
  - Already completed:        106
  - Remaining to compute:     12
  - Checkpoint save interval: Every 20 processed days
  - Output location:          /content/drive/Othercomputers/My Computer/Files_win10/python/py311/stocks/notebooks_RLVR_v2/output/alpha_cache_10d_2026.parquet
  [FINAL SAVED] Flushed and saved remaining progress. Total days: 118
[OK] AlphaCache building phase completed. Final Shape: (110835, 11)


In [ ]:
# CELL 5: Timing
end_time = time.time()
print(f"Time: {time.strftime('%H:%M:%S', time.gmtime(end_time - start_time))}")

Time: 00:01:54
